# 02 · Limpieza: BASE ASISTENCIA A EVENTOS 2026 - EAFIT_anonimizada

Asistencia de empresas a los eventos de ACTHO en 2026 (34 eventos, enero a octubre).

- Entrada: `datos/originales/BASE ASISTENCIA A EVENTOS 2026 - EAFIT_anonimizada.xlsx`
  - Hoja `GENERAL (2)`: matriz empresa × evento con el **número de asistentes** de cada empresa.
  - Hoja `HOMOLOGACION OFERTA 2027`: catálogo de los 34 eventos (fecha, nombre, línea de oferta, modalidad).
- Salidas:
  - `datos/limpios/02_asistencia_largo.csv`: una fila por empresa y evento, con los datos del evento. **Es la que se conecta en Looker Studio.**
  - `datos/limpios/02_asistencia_empresas_limpio.csv`: una fila por empresa, columnas `asistencia_1` … `asistencia_34` y `total_asistencias`.
  - `datos/limpios/02_eventos_catalogo.csv`: una fila por evento. Se une con la anterior por `n_evento` (el número de `asistencia_n`).

Las demás hojas (una por evento) tienen el detalle por persona. No se usan aquí porque `GENERAL (2)` ya las consolida.

In [ ]:
from pathlib import Path

import pandas as pd

BASE = Path.cwd().parent
ORIGINALES = BASE / "datos" / "originales"
LIMPIOS = BASE / "datos" / "limpios"
LIMPIOS.mkdir(exist_ok=True)

ARCHIVO = ORIGINALES / "BASE ASISTENCIA A EVENTOS 2026 - EAFIT_anonimizada.xlsx"

## 1. Carga de la matriz
La hoja `GENERAL (2)` no es una tabla simple:
- Filas 1 a 4: mes, número, fecha y nombre de cada evento (esa información se toma del catálogo, en el paso 5).
- Fila 5: encabezados (`ID_EMPRESA (antes NIT)`, `ASISTENCIA 1` …).
- Filas 6 en adelante: una empresa por fila. Las 2 últimas filas son una fila vacía y una fila de **totales**.
- A la derecha hay columnas de resumen (`TOTAL ASISTENCIAS` y una tabla de distribución) que no son datos.

Se lee sin encabezados y se toma la columna 1 (empresa) y las 34 columnas de eventos. `ASISTENCIA 35` está vacía (no hubo evento 35) y se descarta.

In [ ]:
crudo = pd.read_excel(ARCHIVO, sheet_name="GENERAL (2)", header=None)
assert crudo.shape == (582, 47), crudo.shape

encabezados = crudo.iloc[5]
assert encabezados[1] == "ID_EMPRESA (antes NIT)" and encabezados[35] == "ASISTENCIA 34"
assert (crudo.iloc[6:, 36].isna() | crudo.iloc[6:, 36].eq(0)).all(), "ASISTENCIA 35 ya no está vacía"

N_EVENTOS = 34
COLS_ASIST = [f"asistencia_{n}" for n in range(1, N_EVENTOS + 1)]

df = crudo.iloc[6:, 1:N_EVENTOS + 2].copy()
df.columns = ["id_empresa"] + COLS_ASIST
df.head()

## 2. Filas que no son empresas
Se quitan las filas sin `id_empresa`: la fila vacía y la fila de totales del final.
**No se guarda una fila de totales en los datos**: en Looker Studio cualquier suma la contaría como otra empresa y duplicaría los resultados. Looker calcula los totales solo.

In [ ]:
df = df[df["id_empresa"].notna()].copy()
df["id_empresa"] = df["id_empresa"].astype(str).str.strip()
assert df["id_empresa"].str.fullmatch(r"EMP-\d{4}").all()
len(df)

## 3. Duplicados
6 empresas aparecen dos veces con **exactamente** los mismos valores, así que son copias. Se elimina la segunda. Si en el futuro una empresa repetida tuviera valores distintos, el `assert` avisa en vez de escoger uno a ciegas.

In [ ]:
antes = len(df)
df = df.drop_duplicates()
print("Filas duplicadas eliminadas:", antes - len(df))
assert df["id_empresa"].is_unique, "Hay empresas repetidas con valores distintos: revisar a mano"
assert len(df) == 568, len(df)

## 4. Valores de asistencia
Los valores son el **número de personas** de la empresa que asistieron (0 a 6), no un sí/no. Se pasan a número entero.
Dos eventos no tienen datos confiables en esta hoja; se dejan como vacío (**sin dato**), no como 0, para que el tablero no muestre "0 asistentes":
- **Evento 6** (Live: Suspensión al alza del salario mínimo): la hoja de homologación dice que no tiene registro de asistentes.
- **Evento 27** (Webinar: Depuración normativa laboral): solo 12 empresas tienen valor en la matriz, pero su hoja propia (`AGOSTO 27`) tiene 201 registros. La columna está incompleta.

`total_asistencias` se recalcula; no se usa el total de la hoja, porque no cuadraba en el 5% de las empresas.

In [ ]:
SIN_DATO = [6, 27]

df[COLS_ASIST] = df[COLS_ASIST].apply(pd.to_numeric, errors="raise")
for n in SIN_DATO:
    df[f"asistencia_{n}"] = pd.NA

otros = [c for c in COLS_ASIST if int(c.split("_")[1]) not in SIN_DATO]
assert df[otros].notna().all().all(), "Hay vacíos inesperados en la matriz"
df[COLS_ASIST] = df[COLS_ASIST].astype("Int64")
assert df[otros].ge(0).all().all()

df["total_asistencias"] = df[COLS_ASIST].sum(axis=1).astype("Int64")
df["eventos_asistidos"] = df[COLS_ASIST].gt(0).sum(axis=1)
print("Empresas sin ninguna asistencia:", (df["total_asistencias"] == 0).sum())
df["total_asistencias"].describe()

## 5. Catálogo de eventos
Viene de la hoja `HOMOLOGACION OFERTA 2027`, validada por ACTHO. Trae, para cada evento, la fecha, el nombre limpio, la línea de oferta 2027 y la modalidad.
Se le agrega `total_asistentes` (suma de la matriz) y `con_datos` (False para los eventos 6 y 27).

In [ ]:
eventos = pd.read_excel(ARCHIVO, sheet_name="HOMOLOGACION OFERTA 2027", header=2, nrows=N_EVENTOS)
eventos.columns = ["n_evento", "hoja_origen", "fecha", "nombre_evento", "linea_oferta_2027",
                   "experiencia", "modalidad"]
eventos["n_evento"] = eventos["n_evento"].astype(int)
eventos["fecha"] = pd.to_datetime(eventos["fecha"])
eventos["mes"] = eventos["fecha"].dt.month

assert eventos["n_evento"].tolist() == list(range(1, N_EVENTOS + 1))
assert eventos["fecha"].notna().all()

eventos["con_datos"] = ~eventos["n_evento"].isin(SIN_DATO)
eventos["total_asistentes"] = df[COLS_ASIST].sum().values
eventos.loc[~eventos["con_datos"], "total_asistentes"] = pd.NA
eventos["total_asistentes"] = eventos["total_asistentes"].astype("Int64")
eventos[["n_evento", "fecha", "nombre_evento", "modalidad", "total_asistentes"]]

## 6. Tabla larga para Looker Studio
Looker Studio no puede convertir la matriz (una columna por evento) en filas sin usar código. Por eso se entrega además una tabla **larga**:
una fila por empresa y evento, con los datos del evento ya pegados (fecha, nombre, línea de oferta, modalidad).
Con esta tabla, en Looker basta con arrastrar campos: asistentes por mes, por modalidad, por línea de oferta o por empresa.
En los eventos 6 y 27 `asistentes` queda vacío, y Looker ignora los vacíos al sumar.

In [ ]:
largo = df.melt(id_vars="id_empresa", value_vars=COLS_ASIST, var_name="n_evento", value_name="asistentes")
largo["n_evento"] = largo["n_evento"].str.split("_").str[1].astype(int)
largo = largo.merge(eventos.drop(columns=["total_asistentes", "hoja_origen"]), on="n_evento", how="left")
largo["asistio"] = largo["asistentes"].gt(0).astype("boolean").mask(largo["asistentes"].isna())

assert len(largo) == len(df) * N_EVENTOS
assert largo["nombre_evento"].notna().all()
assert largo["asistentes"].sum() == df["total_asistencias"].sum()
largo.head()

## 7. Guardar
CSV en `utf-8-sig` para que Excel y Looker Studio lean bien las tildes. Se guardan tres archivos:
- `02_asistencia_largo.csv`: **el que se conecta en Looker Studio**.
- `02_asistencia_empresas_limpio.csv`: la matriz por empresa (útil para revisar en Excel).
- `02_eventos_catalogo.csv`: los 34 eventos con su total de asistentes.

In [ ]:
salida = LIMPIOS / "02_asistencia_empresas_limpio.csv"
salida_eventos = LIMPIOS / "02_eventos_catalogo.csv"
salida_largo = LIMPIOS / "02_asistencia_largo.csv"
df.to_csv(salida, index=False, encoding="utf-8-sig")
eventos.to_csv(salida_eventos, index=False, encoding="utf-8-sig")
largo.to_csv(salida_largo, index=False, encoding="utf-8-sig")

assert pd.read_csv(salida).shape == (568, 37)
assert pd.read_csv(salida_eventos).shape == (34, 10)
assert pd.read_csv(salida_largo).shape == (568 * N_EVENTOS, 11)
salida_largo.name, salida.name, salida_eventos.name